## BADR vs SLSQP scalability
This notebook loads the experiment results and visualizes how BADR and SLSQP runtimes scale over time.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

EPSILON = 1e-3  # tolerance for (1 + eps) * best value

notebook_dir = Path.cwd().resolve()
project_root = (
    notebook_dir.parents[1] if len(notebook_dir.parents) > 1 else notebook_dir
)
results_path = project_root / "experiments" / "scalability" / "badr_results.json"
if not results_path.exists():
    raise FileNotFoundError(f"Could not locate BADR results at {results_path}")

with results_path.open("r") as fp:
    raw_results: Dict[str, Dict[str, Dict[str, dict]]] = json.load(fp)


def first_time_within_eps(trace: dict, threshold: float) -> Optional[float]:
    values = trace.get("implicit_value") or []
    cumulative = trace.get("cumulative_seconds") or []
    for value, timestamp in zip(values, cumulative):
        if value <= threshold:
            return float(timestamp)
    return None


records: List[dict] = []
for size_str, runs in raw_results.items():
    sample_size = int(size_str)
    for run_name, alg_results in runs.items():
        available_values: List[float] = []
        for algo_key in ("slsqp", "badr"):
            trace = alg_results.get(algo_key)
            if trace and trace.get("implicit_value"):
                available_values.extend(trace["implicit_value"])
        if not available_values:
            continue
        best_value = min(available_values)
        factor = 1 + EPSILON if best_value >= 0 else 1 - EPSILON
        threshold = best_value * factor

        for algo_key in ("slsqp", "badr"):
            trace = alg_results.get(algo_key)
            if not trace:
                continue
            cumulative = trace.get("cumulative_seconds") or []
            if not cumulative:
                continue
            time_to_eps = first_time_within_eps(trace, threshold)
            records.append(
                {
                    "sample_size": sample_size,
                    "run": run_name,
                    "algorithm": algo_key.upper(),
                    "iterations": len(cumulative),
                    "total_seconds": cumulative[-1],
                    "best_value": best_value,
                    "threshold": threshold,
                    "epsilon": EPSILON,
                    "time_to_eps": time_to_eps,
                }
            )

scalability_df = (
    pd.DataFrame(records)
    .sort_values(["algorithm", "sample_size", "run"])
    .reset_index(drop=True)
)
scalability_df.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tueplots import axes as tp_axes
from tueplots import figsizes

with plt.rc_context(figsizes.jmlr2001(nrows=1, ncols=1)):
    plt.rcParams.update(tp_axes.tick_direction(x="out", y="out"))
    plt.rcParams["font.family"] = "Open Sans"
    plt.rcParams["font.weight"] = "light"
    plt.rcParams.update(
        {
            # "text.usetex": True,
            "font.family": "sans-serif",
            "font.sans-serif": ["Open Sans"],
        }
    )
    plt.rcParams["font.size"] = 9.95
    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["figure.dpi"] = 200
    # Drop incomplete runs and summarize solver statistics
    time_df = scalability_df.dropna(subset=["time_to_eps"]).copy()
    if time_df.empty:
        raise ValueError(
            "No solver reached the (1 + eps) threshold; cannot plot scalability."
        )

    summary_df = (
        time_df.groupby(["algorithm", "sample_size"], as_index=False)
        .agg(
            mean_time=("time_to_eps", "mean"),
            sem_time=("time_to_eps", lambda x: x.sem(ddof=1)),
            runs=("time_to_eps", "size"),
        )
        .sort_values("sample_size")
    )
    summary_df["sem_time"] = summary_df["sem_time"].fillna(0.0)

    plt.figure()
    for algo, group in summary_df.groupby("algorithm"):
        lower = (group["mean_time"] - group["sem_time"]).clip(lower=1e-12)
        upper = (group["mean_time"] + group["sem_time"]).clip(lower=1e-12)
        label = "badr-SGD" if algo == "BADR" else algo

        if algo == "BADR":
            plt.loglog(
                group["sample_size"],
                group["mean_time"],
                marker="o",
                markersize=3,
                label=label,
                zorder=100,
                color="#C81919",
            )
            plt.fill_between(
                group["sample_size"],
                lower,
                upper,
                alpha=0.2,
                zorder=100,
                color="#C81919",
            )
        else:
            plt.loglog(
                group["sample_size"],
                group["mean_time"],
                marker="s",
                markersize=3,
                label=label,
                zorder=1,
                color="#662C91",
            )
            plt.fill_between(
                group["sample_size"], lower, upper, alpha=0.2, zorder=1, color="#662C91"
            )
    plt.title("Runtime", fontsize=13)
    plt.xlabel("Sample size")
    plt.ylabel("Time (seconds)")
    plt.legend(title="Algorithm")
    plt.grid(alpha=0.3, linestyle="-")
    plt.savefig("../../figures/badr_scales.pdf", bbox_inches="tight")
    plt.show()

## Two-loop scalability

In [ ]:
import os
import pickle

import numpy as np
import matplotlib.pyplot as plt
from tueplots import axes as tp_axes
from tueplots import figsizes


def find_index_below_threshold(lst, threshold):
    return next((i for i, v in enumerate(lst) if v < threshold), -1)


def nansem(a, axis=0):
    """Standard error of the mean with NaNs ignored."""
    a = np.asarray(a, dtype=float)
    n = np.sum(~np.isnan(a), axis=axis)
    s = np.nanstd(a, axis=axis, ddof=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return s / np.sqrt(n)


# ---- Load results (.pkl) ----
pkl_path = os.path.join("group_scalability.pkl")
with open(pkl_path, "rb") as f:
    results = pickle.load(f)

states = list(results.keys())
n_groups = sorted({g for s in states for g in results[s].keys()})

# ---- Compute time-to-eps per solver/state/group ----
eps = 1e-5
solvers = ["SLSQP", "FrankWolfe", "TrustConstr"]

time_data = {solver: {state: [] for state in states} for solver in solvers}

for state in states:
    for g in n_groups:
        for solver in solvers:
            hist_f = results[state][g][solver]["history_f"]
            hist_t = results[state][g][solver]["history_time"]
            idx = find_index_below_threshold(hist_f, eps)
            time_data[solver][state].append(hist_t[idx] if idx != -1 else np.nan)

# ---- Aggregate across states: mean + SEM + runs ----
stats = {}
for solver in solvers:
    all_times = np.array(
        [time_data[solver][s] for s in states], dtype=float
    )  # (n_states, n_groups)
    stats[solver] = {
        "mean": np.nanmean(all_times, axis=0),
        "sem": nansem(all_times, axis=0),
        "runs": np.sum(
            ~np.isnan(all_times), axis=0
        ),  # number of successful states per n_groups
    }

# ---- Plot (same style; add bottom success-rate panel) ----
os.makedirs("figures", exist_ok=True)
outpath = os.path.join("../../figures", "scalability_per_group.pdf")

with plt.rc_context(figsizes.jmlr2001(nrows=1, ncols=1)):
    plt.rcParams.update(tp_axes.tick_direction(x="out", y="out"))
    plt.rcParams["font.family"] = "Open Sans"
    plt.rcParams["font.weight"] = "light"
    plt.rcParams.update(
        {
            # "text.usetex": True,
            "font.family": "sans-serif",
            "font.sans-serif": ["Open Sans"],
        }
    )
    plt.rcParams["font.size"] = 9.95
    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["figure.dpi"] = 200

    fig, (ax_time, ax_succ) = plt.subplots(
        2,
        1,
        sharex=True,
        figsize=(8.5, 4.0),
        gridspec_kw={"height_ratios": (1.0, 0.5), "hspace": 0.03},
    )

    plot_cfg = {
        "FrankWolfe": dict(
            label="Frank-Wolfe", marker="o", markersize=3, color="#6699CC", zorder=2
        ),
        "SLSQP": dict(
            label="SLSQP", marker="s", markersize=3, color="#662C91", zorder=100
        ),
        "TrustConstr": dict(
            label="TrustConstr", marker="^", markersize=3, color="#A0AF63", zorder=1
        ),
    }

    x = np.array(n_groups, dtype=float)

    # --- Top: runtime mean ± SEM ---
    for solver in solvers:
        cfg = plot_cfg[solver]
        mean = np.asarray(stats[solver]["mean"], dtype=float)
        sem = np.asarray(stats[solver]["sem"], dtype=float)

        lower = (mean - sem).clip(min=0.0)
        upper = (mean + sem).clip(min=0.0)

        ax_time.plot(
            x,
            mean,
            marker=cfg["marker"],
            markersize=cfg["markersize"],
            label=cfg["label"],
            zorder=cfg["zorder"],
            color=cfg["color"],
        )
        ax_time.fill_between(
            x, lower, upper, alpha=0.2, zorder=cfg["zorder"], color=cfg["color"]
        )

    ax_time.set_title("Runtime")
    ax_time.set_ylabel("Time (seconds)")
    ax_time.legend(title="Solver")
    ax_time.grid(alpha=0.3, linestyle="-")

    # --- Bottom: success rate (uses stats[solver]["runs"]) ---
    denom = float(len(states))
    for solver in solvers:
        cfg = plot_cfg[solver]
        success_rate = np.asarray(stats[solver]["runs"], dtype=float) / denom
        ax_succ.step(
            x,
            success_rate,
            where="mid",
            zorder=cfg["zorder"],
            color=cfg["color"],
            linewidth=2,
        )

    ax_succ.set_ylabel("Success Rate")
    ax_succ.set_xlabel("Number of Groups")
    ax_succ.set_ylim(0.0, 1.05)
    ax_succ.grid(alpha=0.3, linestyle="-")

    fig.savefig(outpath, bbox_inches="tight")
    plt.show()

print(f"Saved: {outpath}")